# 12 — FastAPI Serving Demo

Start the text-to-SQL API server on Colab and test it.
This notebook demonstrates the serving endpoint from `src/serving/app.py`.

**Runtime**: Google Colab T4 GPU

## 0. Setup

In [1]:
!pip install -q transformers peft bitsandbytes accelerate torch
!pip install -q fastapi uvicorn requests

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.9 MB/s eta 0:00:00


In [2]:
!git clone https://github.com/oLittle-five/text-to-sql-llama.git
%cd text-to-sql-llama

Cloning into 'text-to-sql-llama'...
remote: Enumerating objects: 157, done.
remote: Counting objects: 100% (157/157), done.
remote: Compressing objects: 100% (99/99), done.
remote: Total 157 (delta 70), reused 136 (delta 49), pack-reused 0 (from 0)
Receiving objects: 100% (157/157), 5.09 MiB | 8.48 MiB/s, done.
Resolving deltas: 100% (70/70), done.
/content/text-to-sql-llama


In [5]:
from huggingface_hub import login
login()

## 1. Start the API Server

We run the server in a background thread so we can test it from the same notebook.

In [6]:
import threading
import uvicorn
import time
import requests

from src.serving.app import app, load_model, model

# Load model once, before starting the server
if model is None:
    print("Loading model...")
    load_model()
    print("Model loaded!")
else:
    print("Model already loaded.")

# Patch out the lifespan so it doesn't reload
from contextlib import asynccontextmanager

@asynccontextmanager
async def no_reload_lifespan(app):
    yield  # model already loaded above

app.router.lifespan_context = no_reload_lifespan

# Start server in background
def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="warning")

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()

# Wait until ready
for i in range(30):
    try:
        resp = requests.get("http://localhost:8000/health")
        if resp.status_code == 200:
            print("Server running at http://localhost:8000")
            break
    except Exception:
        pass
    time.sleep(1)
else:
    print("Warning: server may not have started properly")

Loading model...
Loading base model: meta-llama/Meta-Llama-3-8B-Instruct
Loading adapter: oLittle-five/llama3-8b-wikisql-qlora
4-bit quantization: True


config.json:   0%|          | 0.00/654 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/73.0 [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/187 [00:00<?, ?B/s]

adapter_config.json: 0.00B [00:00, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/168M [00:00<?, ?B/s]

Model loaded on cuda:0
Model loaded!
Server running at http://localhost:8000


## 2. Test the API

In [7]:
import json

BASE_URL = "http://localhost:8000"

# Health check
resp = requests.get(f"{BASE_URL}/health")
print("Health check:")
print(json.dumps(resp.json(), indent=2))

Health check:
{
  "status": "ok",
  "model": "meta-llama/Meta-Llama-3-8B-Instruct",
  "adapter": "oLittle-five/llama3-8b-wikisql-qlora",
  "quantization": "4-bit NF4",
  "device": "cuda:0"
}


In [8]:
# Single prediction
resp = requests.post(f"{BASE_URL}/predict", json={
    "question": "How many people live in Tokyo?",
    "columns": ["city", "country", "population"],
    "types": ["text", "text", "real"],
})
result = resp.json()
print(f"SQL: {result['sql']}")
print(f"Time: {result['generation_time_ms']:.0f}ms")
print(f"Tokens: {result['prompt_tokens']} prompt + {result['generated_tokens']} generated")

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


SQL: SELECT COUNT(`population`) FROM table WHERE `city` = 'tokyo'
Time: 6172ms
Tokens: 34 prompt + 17 generated


In [9]:
# Batch prediction
resp = requests.post(f"{BASE_URL}/predict/batch", json={
    "queries": [
        {
            "question": "What is the nationality of Terrence Ross?",
            "columns": ["Player", "No.", "Nationality", "Position", "Years in Toronto", "School/Club Team"],
            "types": ["text", "text", "text", "text", "text", "text"],
        },
        {
            "question": "How many schools or teams had Jalen Rose?",
            "columns": ["Player", "No.", "Nationality", "Position", "Years in Toronto", "School/Club Team"],
            "types": ["text", "text", "text", "text", "text", "text"],
        },
        {
            "question": "What was the date of the race in Misano?",
            "columns": ["No", "Date", "Circuit", "Pole Position", "Race winner"],
            "types": ["real", "text", "text", "text", "text"],
        },
    ]
})
result = resp.json()
for i, r in enumerate(result["results"]):
    print(f"Q{i+1}: {r['sql']}  ({r['generation_time_ms']:.0f}ms)")
print(f"\nTotal: {result['total_time_ms']:.0f}ms")

Q1: SELECT `Nationality` FROM table WHERE `Player` = 'terrence ross'  (2625ms)
Q2: SELECT COUNT(`School/Club Team`) FROM table WHERE `Player` = 'Jalen Rose'  (2891ms)
Q3: SELECT `Date` FROM table WHERE `Circuit` ='misano'  (2406ms)

Total: 7925ms


In [10]:
# Validation error test
resp = requests.post(f"{BASE_URL}/predict", json={
    "question": "test",
    "columns": ["col1", "col2"],
    "types": ["text"],  # mismatched length
})
print(f"Status: {resp.status_code}")
print(f"Error: {resp.json()['detail']}")

Status: 400
Error: columns and types must have the same length (got 2 vs 1)


## 3. Example: Interactive Query

Try your own questions!

In [15]:
# Change these to try your own queries!
question = "Which country has the highest GDP?"
columns = ["Country", "GDP (billions)", "Population", "Continent"]
types = ["text", "real", "real", "text"]

resp = requests.post(f"{BASE_URL}/predict", json={
    "question": question,
    "columns": columns,
    "types": types,
})
result = resp.json()
print(f"Question: {question}")
print(f"Columns:  {', '.join(columns)}")
print(f"SQL:      {result['sql']}")
print(f"Time:     {result['generation_time_ms']:.0f}ms")

Question: Which country has the highest GDP?
Columns:  Country, GDP (billions), Population, Continent
SQL:      SELECT MAX(`GDP (billions)`) FROM table WHERE `Continent` = 'asia' AND `Country` = 'indonesia'
Time:     4242ms


## 4. API Documentation

The API auto-generates interactive documentation:
- **Swagger UI**: http://localhost:8000/docs
- **ReDoc**: http://localhost:8000/redoc

(These are accessible when running locally but not from Colab's browser.
The `curl` examples below show the raw API usage.)

In [13]:
# curl-equivalent examples for README
print("""Example curl commands (for when running locally or deployed):

# Health check
curl http://localhost:8000/health

# Single prediction
curl -X POST http://localhost:8000/predict \\
  -H "Content-Type: application/json" \\
  -d '{
    "question": "How many people live in Tokyo?",
    "columns": ["city", "country", "population"],
    "types": ["text", "text", "real"]
  }'

# Batch prediction
curl -X POST http://localhost:8000/predict/batch \\
  -H "Content-Type: application/json" \\
  -d '{"queries": [{"question": "...", "columns": [...], "types": [...]}]}'
""")

Example curl commands (for when running locally or deployed):

# Health check
curl http://localhost:8000/health

# Single prediction
curl -X POST http://localhost:8000/predict \
  -H "Content-Type: application/json" \
  -d '{
    "question": "How many people live in Tokyo?",
    "columns": ["city", "country", "population"],
    "types": ["text", "text", "real"]
  }'

# Batch prediction
curl -X POST http://localhost:8000/predict/batch \
  -H "Content-Type: application/json" \
  -d '{"queries": [{"question": "...", "columns": [...], "types": [...]}]}'

